In [1]:
import os, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm

import albumentations as A
from albumentations.pytorch import ToTensorV2
from pytorch_grad_cam import EigenCAM

warnings.filterwarnings('ignore')
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")

# ── Configuration & Paths ──
SEVERITY_DENSENET_CKPT = '/kaggle/input/notebooks/nandini143/densenet-121-ordinal-severity/densenet_severity/densenet_severity_best.pth'
SEVERITY_EFFNET_CKPT   = '/kaggle/input/notebooks/nandini143/efficientnetv2-ordinal-severity/effnet_severity/effnet_severity_best.pth'
SEVERITY_SWIN_CKPT     = '/kaggle/input/notebooks/nandini143/swin-tiny-ordinal-severity/swin_severity/swin_severity_best.pth'

SEVERITY_CSV    = '/kaggle/input/datasets/nandini143/multiclass-output/severity_data/severity_train_folds.csv'
SEVERITY_IMGDIR = '/kaggle/input/datasets/nandini143/multiclass-output/severity_data/preprocessed_images'

OUTPUT_DIR = '/kaggle/working/xai_outputs'
MULTI_OUT  = os.path.join(OUTPUT_DIR, 'multiclass')

# Ensure output directories exist
for sub in ['heatmaps_densenet', 'heatmaps_efficientnet', 'heatmaps_swin']:
    os.makedirs(os.path.join(MULTI_OUT, sub), exist_ok=True)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE_CNN  = 384
IMG_SIZE_SWIN = 224
SEVERITY_NUM_OUTPUTS = 3

Device : cuda


In [2]:
print("Loading Metadata and Setting up DataFrames...")
sev_df = pd.read_csv(SEVERITY_CSV)
site_cols = [c for c in sev_df.columns if c.startswith('site_')]
actual_meta_dim = 3 + len(site_cols) 

def get_severity_img_path(image_id):
    return os.path.join(SEVERITY_IMGDIR, f'{image_id}.jpg')

sev_df['img_path_resolved'] = sev_df['image'].apply(get_severity_img_path)

class SeverityXAIDataset(Dataset):
    def __init__(self, df, transform, site_cols):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.site_cols = site_cols
        
    def __len__(self): return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(str(row['img_path_resolved']))
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            img = self.transform(image=img)['image']
            
        meta_vals  = [float(row.get('age_norm', 0.5)), float(row.get('sex_male', 0.0)), float(row.get('sex_female', 0.0))]
        meta_vals += [float(row.get(c, 0.0)) for c in self.site_cols]
        meta       = torch.tensor(meta_vals, dtype=torch.float32)
        target_cls = int(row.get('ordinal_label', 3))
        
        return img, meta, row['image'], target_cls

sev_transform_cnn = A.Compose([
    A.Resize(IMG_SIZE_CNN, IMG_SIZE_CNN),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

sev_transform_swin = A.Compose([
    A.Resize(IMG_SIZE_SWIN, IMG_SIZE_SWIN),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])
print(f"Severity Pipeline Ready. Images to process: {len(sev_df)}")

Loading Metadata and Setting up DataFrames...
Severity Pipeline Ready. Images to process: 9340


In [3]:
class TabularMLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 64, out_dim: int = 32, dropout: float = 0.2, use_gelu: bool = True):
        super().__init__()
        act = nn.GELU() if use_gelu else nn.ReLU(inplace=True)
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), act,
            nn.Dropout(dropout), nn.Linear(hidden_dim, out_dim), act,
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor: return self.net(x)

class MetadataSeverityModel(nn.Module):
    def __init__(self, model_name: str, num_outputs: int, meta_dim: int, pretrained: bool = False, drop_rate: float = 0.3, drop_path_rate: float = 0.0, use_gelu: bool = True):
        super().__init__()
        kwargs = dict(pretrained=pretrained, num_classes=0, drop_rate=drop_rate)
        if drop_path_rate > 0: kwargs['drop_path_rate'] = drop_path_rate
        self.backbone = timm.create_model(model_name, **kwargs)
        feat_dim = self.backbone.num_features
        self.use_meta = meta_dim > 0
        if self.use_meta:
            self.meta_mlp = TabularMLP(meta_dim, hidden_dim=64, out_dim=32, dropout=0.2, use_gelu=use_gelu)
            fusion_dim = feat_dim + 32
        else: fusion_dim = feat_dim
            
        norm = nn.BatchNorm1d(fusion_dim) if 'densenet' in model_name else nn.LayerNorm(fusion_dim)
        act  = nn.ReLU(inplace=True) if 'densenet' in model_name else nn.GELU()
        self.head = nn.Sequential(
            norm, nn.Linear(fusion_dim, 512 if 'densenet' in model_name else 256), act,
            nn.Dropout(drop_rate), nn.Linear(512 if 'densenet' in model_name else 256, num_outputs),
        )
        
    def forward(self, images, meta=None):
        feat = self.backbone(images)
        if self.use_meta and meta is not None:
            feat = torch.cat([feat, self.meta_mlp(meta)], dim=1)
        return self.head(feat)

class SwinSeverityModel(nn.Module):
    def __init__(self, model_name: str, num_outputs: int, meta_dim: int, pretrained: bool = False, drop_rate: float = 0.2, drop_path_rate: float = 0.2):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, drop_rate=drop_rate, drop_path_rate=drop_path_rate)
        feat_dim = self.backbone.num_features 
        self.use_meta = meta_dim > 0
        if self.use_meta:
            self.meta_mlp = TabularMLP(meta_dim, 64, 32, 0.2, use_gelu=True)
            fusion_dim = feat_dim + 32
        else: fusion_dim = feat_dim
            
        self.head = nn.Sequential(
            nn.LayerNorm(fusion_dim), nn.Linear(fusion_dim, 512), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(512, 128), nn.GELU(),
            nn.Dropout(0.1), nn.Linear(128, num_outputs),
        )
        
    def forward(self, images, meta=None):
        feat = self.backbone(images)
        if self.use_meta and meta is not None:
            feat = torch.cat([feat, self.meta_mlp(meta)], dim=1) # Fixed from Notebook 1
        return self.head(feat)

# ── Auto-Patching Checkpoint Loader ──
def load_severity_model_safely(builder_fn, ckpt_path, device, meta_dim):
    model = builder_fn(meta_dim=meta_dim)
    ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = ckpt.get('model_state', ckpt.get('model_state_dict', ckpt))
    
    # ── THE FIX: Intercept architecture mismatches before loading ──
    if 'head.4.weight' in state:
        ckpt_shape = state['head.4.weight'].shape
        model_shape = model.head[4].weight.shape
        
        if ckpt_shape != model_shape:
            print(f"⚠️ Mismatch detected! Modifying model architecture to fit checkpoint shape: {ckpt_shape}")
            fusion_dim = model.head[1].in_features
            
            # Identify which activation to use
            if isinstance(model, MetadataSeverityModel) and 'densenet' in str(type(model.backbone)).lower():
                act = nn.ReLU(inplace=True)
            else:
                act = nn.GELU()
            
            if ckpt_shape[0] == 256: 
                # Checkpoint has an intermediate layer of 256. See if there's a head.6 mapped to final outputs
                final_out = state.get('head.6.weight', torch.zeros(3, 256)).shape[0]
                model.head = nn.Sequential(
                    model.head[0], # Norm
                    nn.Linear(fusion_dim, 512), act,
                    nn.Dropout(0.3), nn.Linear(512, 256), act,
                    nn.Dropout(0.1), nn.Linear(256, final_out)
                )
            else:
                # Fallback: Just force the final layer to match the checkpoint
                model.head[4] = nn.Linear(model.head[4].in_features, ckpt_shape[0])

    model.load_state_dict(state, strict=False)
    model.to(device).eval()
    return model

def build_severity_densenet(meta_dim): return MetadataSeverityModel('densenet121', SEVERITY_NUM_OUTPUTS, meta_dim)
def build_severity_efficientnet(meta_dim): return MetadataSeverityModel('tf_efficientnetv2_s', SEVERITY_NUM_OUTPUTS, meta_dim)
def build_severity_swin(meta_dim): return SwinSeverityModel('swin_tiny_patch4_window7_224', SEVERITY_NUM_OUTPUTS, meta_dim)

def find_severity_target_layer(model, model_type: str):
    if model_type == 'densenet': return model.backbone.features.denseblock4
    elif model_type == 'efficientnet': return model.backbone.conv_head
    elif model_type == 'swin': return model.backbone.layers[-1].blocks[-1]
    return None

In [4]:
def save_heatmap(heatmap, out_dir, image_id):
    """Saves a heatmap array as .npy file. Defined here for use by downstream cells."""
    import os
    import numpy as np
    os.makedirs(out_dir, exist_ok=True)
    np.save(os.path.join(out_dir, f"{image_id}.npy"), heatmap.astype(np.float32))

# ─── CELL 5 : XAI Engine — FIXED GradCAM++, Score-CAM & Attention Rollout ───
import torch.nn.functional as F
import numpy as np

class RobustGradCAMPlusPlus:
    """
    Fixed GradCAM++ with Binary Logit Flip.
    For BINARY models (1 output node):
      - If model predicts Malignant (prob >= 0.5): backprop on the Malignant logit.
      - If model predicts Benign   (prob  < 0.5): backprop on NEGATED logit.
        This forces gradients to highlight BENIGN features instead of empty maps.
    For MULTICLASS / CORAL models: uses target_class index as before.
    """
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients   = None
        self._fwd_hook = target_layer.register_forward_hook(self._save_activation)
        self._bwd_hook = target_layer.register_full_backward_hook(self._save_gradient)

    def _reshape_transform(self, x):
        if len(x.shape) == 3:
            B, L, C = x.shape
            H = W = int(np.sqrt(L))
            return x.transpose(1, 2).reshape(B, C, H, W)
        return x

    def _save_activation(self, module, input, output):
        self.activations = self._reshape_transform(output.detach())

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = self._reshape_transform(grad_output[0].detach())

    def generate(self, input_tensor, meta=None, target_class=None, n_smooth=5, noise_sigma=0.02):
        smooth_cams = []
        for _ in range(n_smooth):
            noise       = torch.randn_like(input_tensor) * noise_sigma
            noisy_input = (input_tensor + noise).detach().requires_grad_(True)
            self.model.zero_grad()

            if target_class is None:
                # BINARY MODEL — apply logit flip for Benign images
                logits = self.model(noisy_input, meta)
                prob   = torch.sigmoid(logits).item()
                score  = logits.squeeze() if prob >= 0.5 else -logits.squeeze()
            else:
                # MULTICLASS / CORAL MODEL
                meta_tmp  = meta if meta is not None else torch.zeros(1, actual_meta_dim, device=noisy_input.device)
                logits    = self.model(noisy_input, meta_tmp)
                logit_idx = max(int(target_class) - 1, 0)
                score     = logits[0, logit_idx]

            score.backward()
            acts, grads = self.activations, self.gradients

            grads_sq    = grads ** 2
            grads_cu    = grads ** 3
            alpha_num   = grads_sq
            alpha_denom = 2 * grads_sq + (acts * grads_cu).sum(dim=(2, 3), keepdim=True) + 1e-8
            alpha   = alpha_num / alpha_denom
            weights = (alpha * F.relu(grads)).sum(dim=(2, 3), keepdim=True)
            cam     = (weights * acts).sum(dim=1)
            cam     = F.relu(cam).squeeze(0).cpu().numpy()

            if cam.max() > 0:
                cam = (cam - cam.min()) / (cam.max() - cam.min())
            smooth_cams.append(cam)

        avg_cam = np.mean(smooth_cams, axis=0)
        if avg_cam.max() > 0:
            avg_cam = (avg_cam - avg_cam.min()) / (avg_cam.max() - avg_cam.min())
        return avg_cam

    def remove_hooks(self):
        self._fwd_hook.remove()
        self._bwd_hook.remove()


class RobustScoreCAM:
    """
    Fixed Score-CAM for EfficientNetV2-S (gradient-free).
    
    PREVIOUS BUG: cam += (score - base_prob) * mask
      Masking the tumour LOWERS score, so score - base_prob is NEGATIVE,
      then ReLU zeroes it → blank heatmap.
    
    FIX: Use class-aware weight:
      Malignant image → weight = score         (high when tumour is visible)
      Benign   image  → weight = (1 - score)   (high when benign features visible)
    """
    def __init__(self, model, target_layer, top_k=64):
        self.model, self.top_k = model, top_k
        self.activations = None
        self._fwd_hook = target_layer.register_forward_hook(self._save)

    def _save(self, module, input, output):
        self.activations = output.detach()

    @torch.no_grad()
    def generate(self, input_tensor, meta=None, target_class=None):
        B, C_in, H, W = input_tensor.shape

        if target_class is None:
            logits       = self.model(input_tensor, meta)
            base_prob    = torch.sigmoid(logits).item()
            is_malignant = (base_prob >= 0.5)
        else:
            meta_tmp     = meta if meta is not None else torch.zeros(1, actual_meta_dim, device=input_tensor.device)
            logits       = self.model(input_tensor, meta_tmp)
            logit_idx    = max(int(target_class) - 1, 0)
            base_prob    = torch.sigmoid(logits[0, logit_idx]).item()
            is_malignant = True

        acts    = self.activations
        norms   = acts.std(dim=(2, 3)).squeeze()
        top_idx = norms.topk(min(self.top_k, acts.shape[1])).indices
        cam     = torch.zeros(H, W, device=input_tensor.device)

        for ch in top_idx:
            mask    = acts[0, ch, :, :].unsqueeze(0).unsqueeze(0)
            mask    = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
            mask_up = F.interpolate(mask, size=(H, W), mode='bilinear', align_corners=False)

            masked_input = input_tensor * mask_up
            if target_class is None:
                score  = torch.sigmoid(self.model(masked_input, meta)).item()
                weight = score if is_malignant else (1.0 - score)
            else:
                out    = self.model(masked_input, meta_tmp)
                weight = torch.sigmoid(out[0, logit_idx]).item()

            cam += weight * mask_up.squeeze()

        cam = F.relu(cam).cpu().numpy()
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam

    def remove_hooks(self):
        self._fwd_hook.remove()
        self.model = None


class AttentionRolloutSwin:
    """
    Attention Rollout for Swin-Tiny Transformer.
    
    WHY: GradCAM++ gave 0.00% Drop-Hit for Swin. Shifted-Window attention
         scrambles spatial gradients across window boundaries, making GradCAM
         mathematically meaningless for Swin.
    
    FIX: Attention Rollout traces the actual attention flow through ALL
         transformer layers by multiplying attention matrices, correctly
         identifying which spatial patches the model focused on.
    """
    def __init__(self, model):
        self.model  = model
        self.attns  = []
        self._hooks = []
        self._register_hooks()

    def _register_hooks(self):
        for name, layer in self.model.named_modules():
            if 'WindowAttention' in type(layer).__name__:
                h = layer.register_forward_hook(self._capture_attn)
                self._hooks.append(h)

    def _capture_attn(self, module, input, output):
        # Attempt to capture stored attention weights from the module
        if hasattr(module, 'attn') and module.attn is not None:
            try:
                self.attns.append(module.attn.detach().cpu().float())
            except Exception:
                pass

    @torch.no_grad()
    def generate(self, input_tensor, meta=None, target_class=None):
        self.attns = []
        B, C, H, W = input_tensor.shape
        _ = self.model(input_tensor, meta)

        if len(self.attns) == 0:
            # Fallback: uniform importance map if no attention weights captured
            return np.ones((H, W), dtype=np.float32) * 0.5

        result = None
        for attn in self.attns:
            attn_avg = attn.mean(dim=1)  # (B, N, N) — average over heads
            eye      = torch.eye(attn_avg.shape[-1])
            attn_aug = attn_avg + eye
            attn_aug = attn_aug / attn_aug.sum(dim=-1, keepdim=True)

            if result is None:
                result = attn_aug
            else:
                try:
                    result = torch.bmm(result, attn_aug)
                except Exception:
                    result = attn_aug  # shape mismatch fallback

        # Take attention from first token to all spatial patches
        mask = result[0, 0, :].numpy()
        grid_size = int(np.sqrt(len(mask)))
        if grid_size * grid_size != len(mask):
            grid_size = max(1, int(len(mask) ** 0.5))
            mask = mask[:grid_size * grid_size]
        mask = mask.reshape(grid_size, grid_size)
        mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)

        import cv2
        mask_up = cv2.resize(mask.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)
        return mask_up

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()
        self._hooks = []



class SwinWrapper(torch.nn.Module):
    def __init__(self, model, meta):
        super().__init__()
        self.model = model
        self.meta = meta
    def forward(self, x):
        return self.model(x, meta=self.meta)

def reshape_transform(tensor):
    import numpy as np
    if tensor.ndim == 4:
        return tensor.permute(0, 3, 1, 2)
    elif tensor.ndim == 3:
        B, L, C = tensor.shape
        H = int(np.sqrt(L))
        return tensor.reshape(B, H, H, C).permute(0, 3, 1, 2)
    return tensor

print('XAI Engine v2 ready.')
print('  RobustGradCAMPlusPlus : binary (logit-flip) + multiclass')
print('  RobustScoreCAM        : EfficientNet (class-aware Score-CAM)')
print('  AttentionRolloutSwin  : Swin-Tiny (replaces broken GradCAM++)')


XAI Engine v2 ready.
  RobustGradCAMPlusPlus : binary (logit-flip) + multiclass
  RobustScoreCAM        : EfficientNet (class-aware Score-CAM)
  AttentionRolloutSwin  : Swin-Tiny (replaces broken GradCAM++)


In [5]:
print('[1/3] Processing Severity DenseNet-121...')
sev_dense_model = load_severity_model_safely(build_severity_densenet, SEVERITY_DENSENET_CKPT, DEVICE, actual_meta_dim)
sev_layer_dense = find_severity_target_layer(sev_dense_model, 'densenet')
sev_gcam_dense  = RobustGradCAMPlusPlus(sev_dense_model, sev_layer_dense)

sev_loader = DataLoader(SeverityXAIDataset(sev_df, sev_transform_cnn, site_cols), batch_size=1)

for imgs, meta, img_ids, target_classes in tqdm(sev_loader, desc='Sev-DenseNet'):
    imgs = imgs.to(DEVICE)
    tc   = int(target_classes[0])
    hmap = sev_gcam_dense.generate(imgs, meta=meta.to(DEVICE), target_class=tc)
    save_heatmap(hmap, os.path.join(MULTI_OUT, 'heatmaps_densenet'), img_ids[0])

sev_gcam_dense.remove_hooks()
del sev_dense_model
gc.collect(); torch.cuda.empty_cache()

[1/3] Processing Severity DenseNet-121...
⚠️ Mismatch detected! Modifying model architecture to fit checkpoint shape: torch.Size([256, 512])


Sev-DenseNet: 100%|██████████| 9340/9340 [40:54<00:00,  3.80it/s]


In [6]:
print('[2/3] Processing Severity EfficientNetV2-S...')
sev_eff_model = load_severity_model_safely(build_severity_efficientnet, SEVERITY_EFFNET_CKPT, DEVICE, actual_meta_dim)
sev_layer_eff = find_severity_target_layer(sev_eff_model, 'efficientnet')
sev_scam_eff  = RobustScoreCAM(sev_eff_model, sev_layer_eff)

sev_loader = DataLoader(SeverityXAIDataset(sev_df, sev_transform_cnn, site_cols), batch_size=1)

for imgs, meta, img_ids, target_classes in tqdm(sev_loader, desc='Sev-EfficientNet'):
    imgs = imgs.to(DEVICE)
    tc   = int(target_classes[0])
    hmap = sev_scam_eff.generate(imgs, meta=meta.to(DEVICE), target_class=tc)
    save_heatmap(hmap, os.path.join(MULTI_OUT, 'heatmaps_efficientnet'), img_ids[0])

sev_scam_eff.remove_hooks()
del sev_eff_model
gc.collect(); torch.cuda.empty_cache()

[2/3] Processing Severity EfficientNetV2-S...


Sev-EfficientNet: 100%|██████████| 9340/9340 [3:01:12<00:00,  1.16s/it]


In [7]:
print('[3/3] Processing Severity Swin-Tiny (FIXED: Attention Rollout)...')
sev_swin_model = load_severity_model_safely(build_severity_swin, SEVERITY_SWIN_CKPT, DEVICE, actual_meta_dim)

# FIX: Use AttentionRolloutSwin instead of RobustGradCAMPlusPlus
# GradCAM++ had 0.00% Drop-Hit for Swin because Shifted-Window attention
# scrambles spatial gradients across window boundaries. Attention Rollout
# traces the actual attention flow through all transformer layers.
target_layers = [sev_swin_model.backbone.layers[-1].blocks[-1]]

out_dir = os.path.join(MULTI_OUT, 'heatmaps_swin')
os.makedirs(out_dir, exist_ok=True)

# Use the same DataLoader pattern as the other severity cells (SeverityXAIDataset)
sev_loader_swin = DataLoader(
    SeverityXAIDataset(sev_df, sev_transform_swin, site_cols),
    batch_size=1, shuffle=False, num_workers=2, pin_memory=True
)

for imgs, meta, img_ids, target_classes in tqdm(sev_loader_swin, desc="Sev-Swin"):
    imgs = imgs.to(DEVICE)
    meta = meta.to(DEVICE)
    tc   = int(target_classes[0])
    wrapped_model = SwinWrapper(sev_swin_model, meta)
    cam = EigenCAM(model=wrapped_model, target_layers=target_layers, reshape_transform=reshape_transform)
    hmap = cam(input_tensor=imgs)[0, :]
    save_heatmap(hmap, out_dir, img_ids[0])

del sev_swin_model; gc.collect(); torch.cuda.empty_cache()
print("Severity Swin heatmaps complete.")


[3/3] Processing Severity Swin-Tiny (FIXED: Attention Rollout)...


Sev-Swin:   0%|          | 0/9340 [00:00<?, ?it/s]


ValueError: invalid literal for int() with base 10: 'ISIC_0000002'

In [ ]:
import shutil

print('=' * 65)
print('  NOTEBOOK 2 — MULTICLASS SEVERITY HEATMAPS COMPLETE')
print('=' * 65)

for sub in ['heatmaps_densenet', 'heatmaps_efficientnet', 'heatmaps_swin']:
    path = os.path.join(MULTI_OUT, sub)
    count = len([f for f in os.listdir(path) if f.endswith('.npy')]) if os.path.exists(path) else 0
    print(f'    {sub:30s}: {count} files')

shutil.make_archive('severity_heatmaps_archive', 'zip', MULTI_OUT)
print("\nArchive created: severity_heatmaps_archive.zip")